# COCO-VQA — Reproducible Demo

**Visual Question Answering** on VQAv2 using CLIP ViT-L/14 + BERT + cross-attention fusion.

This notebook evaluates the trained model on **25 curated validation images** bundled in `demo_images/`  
and compares it against a text-only DistilBERT baseline.

---

## How to run

### Option A — Locally (recommended if you have the zip)
```
1. Unzip the project
2. cd coco_vqa
3. pip install -r requirements.txt   (or activate coco_vqa_env)
4. Run all cells
```

### Option B — Google Colab
```
1. Upload coco_vqa.zip to your Google Drive
2. Open this notebook in Colab
3. Set DRIVE_ZIP_PATH in Cell 2 to point to your zip
4. Set GDRIVE_CKPT_ID in Cell 4 to your shared checkpoint file ID
5. Run all cells
```

> **Note:** The main model checkpoint (`best_model.pt`, ~800MB) is too large to include in the zip.  
> It must be downloaded from Google Drive — see Cell 4.

## Cell 1 — Environment detection

In [ ]:
import sys, os
from pathlib import Path

try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

print(f"Running in: {'Google Colab' if IN_COLAB else 'local Jupyter'}")

## Cell 2 — Colab setup (skip if running locally)

In [ ]:
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")

    # ← UPDATE: path to coco_vqa.zip in your Google Drive
    DRIVE_ZIP_PATH = "/content/drive/MyDrive/coco_vqa.zip"

    import zipfile
    if not Path("/content/coco_vqa").exists():
        print(f"Unzipping {DRIVE_ZIP_PATH} ...")
        with zipfile.ZipFile(DRIVE_ZIP_PATH) as z:
            z.extractall("/content")
        print("Done.")

    os.chdir("/content/coco_vqa")
    print(f"Working directory: {os.getcwd()}")
else:
    # Local: notebook should already be at project root
    print(f"Working directory: {os.getcwd()}")

## Cell 3 — Install dependencies

In [ ]:
# Only runs if packages are missing — safe to run even if already installed
%pip install -q torch torchvision transformers Pillow gdown pyyaml matplotlib

## Cell 4 — Imports + project path

In [ ]:
import json
import random
import warnings
from collections import Counter
from pathlib import Path
from typing import Dict, List, Tuple

import torch
import torch.nn as nn
import yaml
import matplotlib.pyplot as plt
from PIL import Image
from transformers import BertTokenizer, DistilBertModel

warnings.filterwarnings("ignore", category=UserWarning)

PROJECT_ROOT = Path(os.getcwd())
sys.path.insert(0, str(PROJECT_ROOT))

assert (PROJECT_ROOT / "src").exists(), (
    f"src/ not found in {PROJECT_ROOT}. "
    "Make sure you are running from the project root."
)

from src.data.answer_vocab import load_vocab
from src.data.augmentations import get_val_transforms
from src.models.vqa_model import VQAModel

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"PyTorch {torch.__version__} | device: {device}")

## Cell 5 — Load config + vocab

In [ ]:
with open(PROJECT_ROOT / "configs/config.yaml") as f:
    cfg = yaml.safe_load(f)

vocab_path = PROJECT_ROOT / cfg["paths"]["vocab_path"]
ans2idx, idx2ans = load_vocab(vocab_path)
print(f"Vocab size:  {len(ans2idx)}")
print(f"Sample answers: {idx2ans[:8]}")

## Cell 6 — Load main model checkpoint

The checkpoint is loaded from `outputs/checkpoints/best_model.pt` if it exists locally.  
Otherwise it is downloaded from Google Drive — **set `GDRIVE_CKPT_ID` to your file ID**.

To get your file ID: open `best_model.pt` in Google Drive → Share → copy the link.  
The ID is the long string between `/d/` and `/view` in the URL.

In [ ]:
CKPT_PATH = PROJECT_ROOT / "outputs/checkpoints/best_model.pt"

if not CKPT_PATH.exists():
    GDRIVE_CKPT_ID = "1QnHOjS2NgMP6tKqmyw80ZDR6D9Qj9D5H"

    import gdown
    CKPT_PATH.parent.mkdir(parents=True, exist_ok=True)
    print("Downloading checkpoint from Google Drive...")
    gdown.download(id=GDRIVE_CKPT_ID, output=str(CKPT_PATH), quiet=False)

checkpoint = torch.load(CKPT_PATH, map_location=device)
print(f"Checkpoint epoch : {checkpoint.get('epoch', '?')}")
print(f"Best val accuracy: {checkpoint.get('best_val_accuracy', '?')}")

## Cell 7 — Initialise main model

In [ ]:
model = VQAModel(
    config=cfg,
    vocab_size=len(ans2idx),
    mode="multimodal",
).to(device)

model.load_state_dict(checkpoint["model_state_dict"], strict=False)
model.eval()

total_params     = sum(p.numel() for p in model.parameters()) / 1e6
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad) / 1e6
print(f"Parameters: {total_params:.1f}M total | {trainable_params:.1f}M trainable")
print("Main model ready.")

## Cell 8 — Load demo images + metadata

In [ ]:
DEMO_DIR  = PROJECT_ROOT / "demo_images"
META_PATH = DEMO_DIR / "metadata.json"

assert META_PATH.exists(), (
    "demo_images/metadata.json not found.\n"
    "Run first:  python scripts/prepare_demo.py"
)

with open(META_PATH, encoding="utf-8") as f:
    metadata = json.load(f)

by_type = Counter(m["answer_type"] for m in metadata)
print(f"Demo samples: {len(metadata)}")
print(f"  yes/no: {by_type.get('yes/no', 0)}  "
      f"number: {by_type.get('number', 0)}  "
      f"other:  {by_type.get('other', 0)}")

## Cell 9 — Inference helpers

In [ ]:
transform  = get_val_transforms(cfg["data"]["image_size"])
tokenizer  = BertTokenizer.from_pretrained(cfg["model"]["text_encoder"])
MAX_LEN    = cfg["data"]["max_question_length"]


def vqa_soft_score(pred: str, gt_answers: List[str]) -> float:
    """VQA v2 soft accuracy: min(#annotators who said pred / 3, 1.0)."""
    return min(Counter(gt_answers)[pred] / 3.0, 1.0)


def predict_main(image_path: Path, question: str) -> Tuple[str, float]:
    """Run main multimodal model. Returns (predicted_answer, confidence)."""
    img_t = transform(Image.open(image_path).convert("RGB")).unsqueeze(0).to(device)
    enc   = tokenizer(
        question, padding="max_length", truncation=True,
        max_length=MAX_LEN, return_tensors="pt",
    )
    with torch.no_grad():
        out = model({
            "image_tensor":  img_t,
            "question_ids":  enc["input_ids"].to(device),
            "attention_mask": enc["attention_mask"].to(device),
        })
    logits   = out["answer_logits"].squeeze(0)
    pred_idx = logits.argmax().item()
    conf     = logits.softmax(dim=-1)[pred_idx].item()
    pred_ans = idx2ans[pred_idx] if pred_idx < len(idx2ans) else "unknown"
    return pred_ans, conf


print("Inference helpers ready.")

## Cell 10 — Run main model on all 25 demo images

In [ ]:
main_results = []
print("Running inference...")

for i, m in enumerate(metadata):
    pred, conf = predict_main(DEMO_DIR / m["filename"], m["question"])
    score = vqa_soft_score(pred, m["answers"])
    main_results.append({**m, "prediction": pred, "confidence": round(conf, 4), "score": score})
    print(f"  [{i+1:02d}/{len(metadata)}] {m['question'][:45]:<45}  "
          f"pred={pred:<12}  gt={m['most_common_answer']:<12}  score={score:.2f}")

overall = sum(r["score"] for r in main_results) / len(main_results)
print(f"\nOverall VQA soft accuracy: {overall:.4f}  ({overall*100:.1f}%)")

## Cell 11 — Per-type accuracy breakdown

In [ ]:
def accuracy_by_type(results: List[dict]) -> Dict[str, float]:
    out = {}
    for at in ["yes/no", "number", "other"]:
        subset = [r for r in results if r["answer_type"] == at]
        out[at] = sum(r["score"] for r in subset) / len(subset) if subset else 0.0
    out["overall"] = sum(r["score"] for r in results) / len(results)
    return out

main_acc = accuracy_by_type(main_results)

SEP = "-" * 48
print(SEP)
print(f"{'Metric':<25} {'Accuracy':>10} {'%':>8}")
print(SEP)
for k, v in main_acc.items():
    print(f"{k:<25} {v:>10.4f} {v*100:>7.1f}%")
print(SEP)

## Cell 12 — Visualise sample predictions

In [ ]:
n_show = min(9, len(main_results))
fig, axes = plt.subplots(3, 3, figsize=(15, 13))
axes = axes.flatten()

for ax, r in zip(axes, main_results[:n_show]):
    ax.imshow(Image.open(DEMO_DIR / r["filename"]))
    ax.axis("off")
    color  = "#1b5e20" if r["score"] >= 0.5 else "#b71c1c"
    marker = "V" if r["score"] >= 0.5 else "X"
    q_text = r["question"] if len(r["question"]) <= 48 else r["question"][:45] + "..."
    ax.set_title(
        f"{marker} {q_text}\n"
        f"Pred: {r['prediction']}   GT: {r['most_common_answer']}",
        fontsize=8, color=color, pad=4,
    )

plt.suptitle(
    f"CLIP + BERT + CrossAttn  |  overall acc = {overall*100:.1f}%",
    fontsize=13, fontweight="bold", y=1.01,
)
plt.tight_layout()
out_png = PROJECT_ROOT / "outputs/demo_predictions.png"
out_png.parent.mkdir(parents=True, exist_ok=True)
plt.savefig(out_png, dpi=130, bbox_inches="tight")
plt.show()
print(f"Saved -> {out_png}")

## Cell 13 — Text-only DistilBERT baseline

This section loads the baseline checkpoint and runs it on the **same 25 images**  
(text only — no image input) for a direct, fair comparison.

If the baseline has not been trained yet, this cell prints a warning and skips.

In [ ]:
class TextOnlyBERT(nn.Module):
    """DistilBERT CLS -> 2-layer MLP -> answer logits (text only, no image)."""
    def __init__(self, text_encoder: str, hidden_dim: int, num_classes: int):
        super().__init__()
        self.bert = DistilBertModel.from_pretrained(text_encoder)
        d = self.bert.config.hidden_size
        self.mlp = nn.Sequential(
            nn.Linear(d, hidden_dim),  nn.ReLU(), nn.Dropout(0.1),
            nn.Linear(hidden_dim, hidden_dim // 2), nn.ReLU(), nn.Dropout(0.1),
            nn.Linear(hidden_dim // 2, num_classes),
        )
    def forward(self, input_ids, attention_mask):
        cls = self.bert(input_ids=input_ids,
                        attention_mask=attention_mask).last_hidden_state[:, 0]
        return self.mlp(cls)


BASELINE_CKPT = (
    PROJECT_ROOT / "baselines/outputs/checkpoints/text_only_bert/best.pt"
)

baseline        = None
baseline_results = None

if not BASELINE_CKPT.exists():
    print("Baseline checkpoint not found — skipping comparison.")
    print(f"To train: python baselines/train_baselines.py "
          f"--config baselines/configs/baselines_config.yaml")
else:
    base_cfg = yaml.safe_load(
        (PROJECT_ROOT / "baselines/configs/baselines_config.yaml").read_text()
    )
    bm_cfg = base_cfg["models"]["text_only_bert"]

    baseline = TextOnlyBERT(
        text_encoder=bm_cfg["text_encoder"],
        hidden_dim=int(bm_cfg["hidden_dim"]),
        num_classes=len(ans2idx),
    ).to(device)

    saved = torch.load(BASELINE_CKPT, map_location=device)
    try:
        baseline.load_state_dict(saved["model_state_dict"])
        baseline.eval()
        print(f"Baseline loaded (epoch {saved.get('epoch','?')}, "
              f"val_acc={saved.get('val_accuracy','?')})")
    except RuntimeError as e:
        print(f"Could not load baseline weights: {e}")
        baseline = None

## Cell 14 — Run baseline inference

In [ ]:
if baseline is not None:
    base_tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")
    base_cfg       = yaml.safe_load(
        (PROJECT_ROOT / "baselines/configs/baselines_config.yaml").read_text()
    )
    BASE_MAX_LEN = int(base_cfg["data"]["max_question_length"])

    baseline_results = []
    for m in metadata:
        enc = base_tokenizer(
            m["question"], padding="max_length", truncation=True,
            max_length=BASE_MAX_LEN, return_tensors="pt",
        )
        with torch.no_grad():
            logits = baseline(
                enc["input_ids"].to(device),
                enc["attention_mask"].to(device),
            )
        pred_idx = logits.argmax(dim=-1).item()
        pred     = idx2ans[pred_idx] if pred_idx < len(idx2ans) else "unknown"
        score    = vqa_soft_score(pred, m["answers"])
        baseline_results.append({**m, "prediction": pred, "score": score})

    base_acc    = accuracy_by_type(baseline_results)
    base_overall = base_acc["overall"]
    print(f"Baseline overall accuracy: {base_overall:.4f}  ({base_overall*100:.1f}%)")
else:
    print("Baseline not available — skipping.")

## Cell 15 — Side-by-side comparison

In [ ]:
if baseline_results is not None:
    cats  = ["Overall", "Yes/No", "Number", "Other"]
    keys  = ["overall", "yes/no", "number", "other"]
    m_scores = [main_acc[k] * 100 for k in keys]
    b_scores = [base_acc[k]  * 100 for k in keys]

    x, w = range(len(cats)), 0.35
    fig, ax = plt.subplots(figsize=(11, 6))
    ax.bar([i - w/2 for i in x], b_scores, w,
           label="Text-Only DistilBERT (baseline)", color="#9e9e9e", edgecolor="white")
    ax.bar([i + w/2 for i in x], m_scores, w,
           label="CLIP + BERT + CrossAttn (main)", color="#1976d2", edgecolor="white")

    for i, (b, m) in enumerate(zip(b_scores, m_scores)):
        ax.annotate(
            f"+{m - b:.1f}%",
            xy=(i + w/2, m), xytext=(0, 5), textcoords="offset points",
            ha="center", fontsize=9, color="#1976d2", fontweight="bold",
        )

    ax.set_xticks(list(x))
    ax.set_xticklabels(cats)
    ax.set_ylabel("VQA Accuracy (%)")
    ax.set_title("Main Multimodal Model vs Text-Only Baseline")
    ax.legend()
    ax.set_ylim(0, min(100, max(m_scores) * 1.35))
    ax.grid(axis="y", alpha=0.3)
    plt.tight_layout()
    out_cmp = PROJECT_ROOT / "outputs/demo_comparison.png"
    plt.savefig(out_cmp, dpi=130, bbox_inches="tight")
    plt.show()
    print(f"Saved -> {out_cmp}")

    # Printed table
    SEP2 = "-" * 68
    print(f"\n{SEP2}")
    print(f"{'Model':<35} {'Overall':>7} {'Yes/No':>7} {'Number':>7} {'Other':>7}")
    print(SEP2)
    print(f"{'Text-Only DistilBERT':<35} "
          + "  ".join(f"{base_acc[k]*100:>6.1f}%" for k in keys))
    print(f"{'CLIP + BERT + CrossAttn':<35} "
          + "  ".join(f"{main_acc[k]*100:>6.1f}%" for k in keys))
    print(SEP2)
    print(f"{'Improvement':<35} "
          + "  ".join(f"{(main_acc[k]-base_acc[k])*100:>+6.1f}%" for k in keys))
    print(SEP2)
else:
    print("Baseline not available — comparison skipped.")
    print("Train the baseline first, then re-run this cell.")

## Cell 16 — Language bias analysis

The text-only baseline reveals how much accuracy is achievable from question phrasing alone,  
without seeing any image. High yes/no accuracy in the baseline = language bias.

In [ ]:
if baseline_results is not None:
    print("Language bias analysis")
    print(f"  Text-only yes/no: {base_acc['yes/no']*100:.1f}%  "
          "(model exploits question phrasing without seeing any image)")
    print(f"  Multimodal yes/no: {main_acc['yes/no']*100:.1f}%  "
          f"(+{(main_acc['yes/no']-base_acc['yes/no'])*100:.1f}% from vision)")
    print(f"  Number gap: +{(main_acc['number']-base_acc['number'])*100:.1f}%  "
          "(counting requires visual grounding — largest expected gain)")
    print()
    # Top predicted answers from baseline (collapse check)
    top_base = Counter(r["prediction"] for r in baseline_results).most_common(5)
    print("Top-5 baseline predictions (collapse indicator):")
    for ans, cnt in top_base:
        print(f"  '{ans}': {cnt}/{len(baseline_results)} ({cnt/len(baseline_results)*100:.0f}%)")
else:
    print("Baseline not available.")

## Cell 18 — Summary

Results saved to `outputs/demo_predictions.png` and `outputs/demo_comparison.png`.

In [ ]:
# ── change these two lines ───────────────────────────────────────────────────
IMAGE_FILE = "COCO_val2014_000000000000.jpg"   # filename from demo_images/
QUESTION   = ""                                # leave blank = use original question
# ─────────────────────────────────────────────────────────────────────────────

# List available filenames
available = sorted(m["filename"] for m in metadata)
if IMAGE_FILE not in available:
    print("Available images:")
    for i, fn in enumerate(available):
        mm = next(x for x in metadata if x["filename"] == fn)
        print(f"  {i:2d}  {fn}  |  {mm['answer_type']:<6}  |  {mm['question']}")
    raise ValueError(f"'{IMAGE_FILE}' not found in demo_images/ — pick one from the list above.")

m   = next(x for x in metadata if x["filename"] == IMAGE_FILE)
q   = QUESTION.strip() if QUESTION.strip() else m["question"]
img = Image.open(DEMO_DIR / m["filename"]).convert("RGB")

# Run inference — get top-5 answers with probabilities
img_t = transform(img).unsqueeze(0).to(device)
enc   = tokenizer(q, padding="max_length", truncation=True,
                  max_length=MAX_LEN, return_tensors="pt")
with torch.no_grad():
    out    = model({"image_tensor":   img_t,
                    "question_ids":   enc["input_ids"].to(device),
                    "attention_mask": enc["attention_mask"].to(device)})
probs   = out["answer_logits"].squeeze(0).softmax(dim=-1)
top5_v, top5_i = probs.topk(5)
top5    = [(idx2ans[i.item()], v.item()) for i, v in zip(top5_i, top5_v)]
pred_ans, pred_conf = top5[0]

# ── plot ─────────────────────────────────────────────────────────────────────
fig, (ax_img, ax_bar) = plt.subplots(1, 2, figsize=(13, 5),
                                      gridspec_kw={"width_ratios": [1, 1.4]})
ax_img.imshow(img)
ax_img.axis("off")
score = vqa_soft_score(pred_ans, m["answers"])
color = "#1b5e20" if score >= 0.5 else "#b71c1c"
ax_img.set_title(
    f"Q: {q}\n"
    f"Pred: {pred_ans}  ({pred_conf*100:.1f}%)   "
    f"GT: {m['most_common_answer']}   score={score:.2f}",
    fontsize=9, color=color, pad=6,
)

answers    = [a for a, _ in top5]
confs      = [c * 100 for _, c in top5]
bar_colors = ["#1976d2" if a == pred_ans else "#90caf9" for a in answers]
bars = ax_bar.barh(answers[::-1], confs[::-1], color=bar_colors[::-1], edgecolor="white")
for bar, conf in zip(bars, confs[::-1]):
    ax_bar.text(bar.get_width() + 0.3, bar.get_y() + bar.get_height() / 2,
                f"{conf:.1f}%", va="center", fontsize=9)
ax_bar.set_xlabel("Confidence (%)")
ax_bar.set_title("Top-5 predicted answers")
ax_bar.set_xlim(0, max(confs) * 1.25)
ax_bar.grid(axis="x", alpha=0.3)
plt.tight_layout()
plt.show()

## Cell 18 — Summary

Results saved to `outputs/demo_predictions.png` and `outputs/demo_comparison.png`.

In [ ]:
print("=" * 55)
print(" REPRODUCE.IPYNB SUMMARY")
print("=" * 55)
print(f" Model          : CLIP ViT-L/14 + BERT + CrossAttn")
print(f" Vocab size     : {len(ans2idx)}")
print(f" Demo samples   : {len(metadata)} "
      f"({by_type.get('yes/no',0)} yes/no | "
      f"{by_type.get('number',0)} number | "
      f"{by_type.get('other',0)} other)")
print(f" Main model acc : {main_acc['overall']*100:.1f}%")
if baseline_results is not None:
    print(f" Baseline acc   : {base_acc['overall']*100:.1f}%")
    print(f" Vision gain    : +{(main_acc['overall']-base_acc['overall'])*100:.1f}%")
print("=" * 55)
print(" Outputs written to outputs/")